In [15]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import brier_score_loss
import joblib

In [18]:
## Basic Logistic Regression with random 80/20 test/train split
## Model v01

# Read data from CSV
data = pd.read_csv('InputData.csv')

# Separating input and output data
X = data.drop(columns=["Team 1", "Team 2", "Date", "Outcome"])
y = data["Outcome"]

# Retain the dropped columns for X_test tracking
game_info = data[["Team 1", "Team 2", "Date"]]

# Splitting data into training and testing data, and also split the game info
X_train, X_test, y_train, y_test, game_info_train, game_info_test = train_test_split(
    X, y, game_info, test_size=0.2, random_state=42
)

# Initializing Model and scaler
scaler = StandardScaler()
model = LogisticRegression()

# Standard scaling data
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Training Model
model.fit(X_train, y_train)

# Saving model
# joblib.dump(model, 'model-v01.joblib')

# Getting prediction probabilities
win_probabilities = model.predict_proba(X_test)[:, 1]

# Combine game info with win probabilities into a DataFrame
results = game_info_test.copy()
results["Win Probability"] = win_probabilities

# results.to_csv('probs.csv', index=False, header=True)

# Calculate Testing Accuracy
test_accuracy = model.score(X_test, y_test)
print(f"Testing Accuracy: {100*test_accuracy:.2f}%")

# Calculate Brier Score
brier = brier_score_loss(y_test, win_probabilities)
print(f"Brier Score: {brier:.4f}")

Testing Accuracy: 64.29%
Brier Score: 0.2146


In [ ]:
## Logistic Regression with PCA
## Model v02

# Read data
data = pd.read_csv('InputData.csv')

# Separate features and target
X = data.drop(columns=["Team 1", "Team 2", "Date", "Outcome"])
y = data["Outcome"]

# Retain game info for tracking
game_info = data[["Team 1", "Team 2", "Date"]]

# Train-test split
X_train, X_test, y_train, y_test, game_info_train, game_info_test = train_test_split(
    X, y, game_info, test_size=0.2, random_state=42
)

# Initialize scaler, PCA, and model
scaler = StandardScaler()
pca = PCA(n_components=0.95)  # Keeps 95% variance, or set to specific int like 10
model = LogisticRegression()

# Scale the data
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Apply PCA
X_train_pca = pca.fit_transform(X_train_scaled)
X_test_pca = pca.transform(X_test_scaled)

# Train model
model.fit(X_train_pca, y_train)

# Save model and PCA
# joblib.dump(model, 'model-v02.joblib')
# joblib.dump(pca, 'pca-v02.joblib')
# joblib.dump(scaler, 'scaler-v02.joblib')

# Predict probabilities
win_probabilities = model.predict_proba(X_test_pca)[:, 1]

# Save results
results = game_info_test.copy()
results["Win Probability"] = win_probabilities
# results.to_csv('probs.csv', index=False, header=True)

# Calculate Testing Accuracy
test_accuracy = model.score(X_test_pca, y_test)
print(f"Testing Accuracy: {100 * test_accuracy:.2f}%")

# Calculate Brier Score
brier = brier_score_loss(y_test, win_probabilities)
print(f"Brier Score: {brier:.4f}")

# Displaying Number of PCA Components
print(f"Number of PCA components: {pca.n_components_}")



Testing Accuracy: 66.67%
Brier Score: 0.2173
Number of PCA components: 21
